# 05 · Enterprise architecture for agent memory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AreevAI/dejadb/blob/main/examples/colab/05_enterprise_architecture.ipynb)

*Segment: "Architectural considerations for enterprise agent systems."*

The governance loop is only credible if the substrate holds up in production.
DejaDB's load-bearing choices, each demonstrated below:

- **Embedded** — the library *is* the database; microsecond recall in-process,
  works air-gapped, nothing to operate per agent.
- **One memory = one file** — the unit of tenancy, encryption, erasure, sync,
  and portability.
- **Immutable + content-addressed** — tampering is detectable, history is
  append-only.
- **Open format** (OMS) — your memories are not hostage to a vendor.

In [1]:
# dejadb 1.0.5 is on PyPI; `dejadb.helpers` ships inside the wheel.
%pip install -q "dejadb>=1.0.5" matplotlib

from dejadb.helpers import *
import dejadb, json, pathlib
print("dejadb", dejadb.__version__)

dejadb 1.0.3


## Tenancy: one file per customer

Isolation by construction — a tenant's memory is a file you can place, encrypt,
copy, or delete as a unit. No shared tables, no row-level security to get
wrong:

In [2]:
acme   = fresh("cust-acme.db",   ns="acme",   actor="agent:support")
globex = fresh("cust-globex.db", ns="globex", actor="agent:support")

acme.add_fact("acme-corp", "plan", "enterprise")
globex.add_fact("globex", "plan", "starter")

print("acme's file knows about globex?", facts(acme, "globex"))
print("files on disk:", sorted(str(p) for p in pathlib.Path(".").glob("cust-*.db")))

acme's file knows about globex? []
files on disk: ['cust-acme.db', 'cust-acme.db.telemetry.db', 'cust-globex.db', 'cust-globex.db.telemetry.db']


## Encryption at rest — and erasure as key deletion

`passphrase=` gives AES-256-GCM at rest (Argon2id-derived key). A wrong key
doesn't return garbage — it refuses. Because the tenant *is* the file,
GDPR-grade erasure is: delete the file and its key. Single-record removal is a
`forget` tombstone:

In [3]:
s = fresh("hr-records.db", ns="hr", passphrase="demo-only-passphrase")
s.add_fact("employee-207", "work_authorization", "verified 2026-03")
note = s.add_fact("employee-207", "note", "temporary accommodation request")
del s

try:
    dejadb.DejaDB("hr-records.db", ns="hr", passphrase="wrong")
except ValueError as e:
    print("wrong key:", str(e)[:58], "…")

s = dejadb.DejaDB("hr-records.db", ns="hr", passphrase="demo-only-passphrase")
s.forget(note)                                     # data-subject request: one record
show_facts(s, "employee-207")
print("integrity:", s.verify())

wrong key: STO-E001: storage error: Decryption failed for page=1 …
  employee-207 --work_authorization--> verified 2026-03
integrity: {"grains":1,"integrity":"ok"}


## Portability & sync: oplog bundles

Files travel. `bundle()` exports the operation log from a cursor;
`import_bundle()` fast-forwards a replica, idempotently — the same mechanism
the `dejad` hub uses to let voice, WhatsApp, and email channels share one
memory (the repo's multichannel acceptance test):

In [4]:
acme.bundle("acme.mgb", 0)
replica = fresh("replica.db", ns="acme")
print("replica applied", replica.import_bundle("acme.mgb"), "ops →")
show_facts(replica, "acme-corp")

replica applied 1 ops →
  acme-corp --plan--> enterprise


[{'relation': 'plan',
  'object': 'enterprise',
  'confidence': 0.9,
  'hash': '367664a17ca25508dc8ab3bb9f2f230703d43e1f6a420e52d6c139b47a96e294'}]

## Migration: bring the memories you already have

Importers for mem0, LangGraph, Letta, Zep, and raw tool-call JSONL — idempotent,
original timestamps kept, provenance attached. A mem0 export:

In [5]:
export = {"results": [
    {"id": "m1", "memory": "Customer prefers invoices in EUR",
     "user_id": "acme", "created_at": "2026-05-01T10:00:00Z"},
    {"id": "m2", "memory": "Escalations should page the on-call TAM",
     "user_id": "acme", "created_at": "2026-06-11T09:30:00Z"},
]}
print("report:", acme.migrate("mem0", json.dumps(export)))

hit = json.loads(acme.cal('RECALL facts ABOUT "invoices" | LIMIT 1'))["grains"][0]["fields"]
print("imported:", hit["context"]["content"])
print("provenance:", hit["context"]["import"], "— original created_at preserved")

report: {"added":2,"forgotten":0,"notes":[],"skipped":0,"superseded":0}
imported: Customer prefers invoices in EUR
provenance: {'id': 'm1', 'source': 'mem0', 'user_id': 'acme'} — original created_at preserved


## Governance as code, per environment

The autonomy surface is a reviewable JSON artifact — dev, team, and locked-down
production variants live in the repo
([examples/policy/](https://github.com/AreevAI/dejadb/tree/main/examples/policy)),
not in a UI:

```jsonc
// locked-down.json — production
{ "auto_apply_enabled": false,                  // nothing self-applies
  "deny": ["waiser.staleness"],                 // this analyzer: off entirely
  "severity_floors": { "waiser.contradiction_sweep": "high" },
  "telemetry": "off" }
```

CI gates on the review queue exactly like it gates on tests:
`deja waiser list --fail-on high` → exit 2 while high-severity findings await.

## The reference deployment

```
 agent loop (your framework / Claude memory-tool / MCP)
   │  recall · record_tool_call · remember          in-process, µs reads
   ▼
 per-tenant .db files  ──►  Waiser sweep (cron / --if-stale / CI)
   │   AES-256 at rest        │  deterministic analyzers (+ optional LLM,
   │   forget / crypto-erase  │  DISCOVER→GROUND→VERIFY, no write access)
   ▼                          ▼
 dejad hub (bundle sync) ◄── review queue ── console / CLI / MCP / HTTP
                              approve · reject · rollback (reason required)
```

Every surface — Python, Node, CLI, MCP, HTTP — speaks the same file; every
user-facing error carries a stable `DOMAIN-Ennn` code your observability stack
can alert on.

In [6]:
print("stats:", acme.stats())
print("file-vs-host reconciliation warnings:", acme.open_warnings())

stats: {"current":3,"events_indexed":0,"grains":3,"ops":3,"terms":9,"triples":3}
file-vs-host reconciliation warnings: []


## Takeaways for architects

1. Put the **memory boundary where the tenancy boundary is** — a file per
   customer/agent beats row filters in a shared store.
2. Make learning **asynchronous and gated** — agents record evidence in the hot
   path; analysis, review, and adoption happen out-of-band.
3. Treat agent knowledge like code: **proposals, reviews, CI gates, rollback**.
4. Demand **measurement** — a lesson that can't be scored `held`/`regressed`
   is a liability, not an improvement.

*The loop itself: notebooks 01–04. Full tour: `self_improving_agents.ipynb`.*